# Estudio del Entrenamiento de Gaussianas

Este notebook permite ejecutar el entrenamiento de 3D Gaussian Splatting de forma interactiva, visualizando los resultados y analizando las métricas de densidad.

In [ ]:
import torch
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from tqdm.auto import tqdm
from opensplat3d.utils.vis_utils import pca, enhance_image
import numpy as np
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

from opensplat3d.utils.train_utils import setup_training, train_cameras
from opensplat3d.scene import Scene
from opensplat3d.gaussian_model import create_from_pcd
from opensplat3d.gaussian_optimizer import GaussianOptimizer
from opensplat3d.gaussian_renderer import render
from opensplat3d.losses import l1_loss, ssim, instance_2d_loss, get_erank_loss, get_thinness_loss
from opensplat3d.data import load_scene_info
from opensplat3d.utils.train_utils import train_cameras

device = torch.device("cuda:0")
device2 = torch.device("cuda:1")

## 1. Configuración y Carga de Datos

Asegúrate de apuntar a un archivo de configuración válido y establecer el path del modelo.

In [ ]:
class Args:
    config = "/home/ubuntu/semantic-gaussians/configs/lerf.yaml"
    overrides = [
        "model.source_path=/home/ubuntu/datasets/Replica/office0",
        "model.model_path=output/training_study_replica",
        "opt.lambda_depth=0.5" 
    ]
    wandb = None
    checkpoint = None
    
args = Args()
config = setup_training(args)

model_params = config.model
opt_params = config.opt
pipe_params = config.pipe

scene_info = load_scene_info(model_params)
scene = Scene(scene_info, model_params.resolution, device)
viewpoint_cams = train_cameras(scene)

gaussians = create_from_pcd(
    scene_info.point_cloud,
    model_params.sh_degree,
    device,
    model_params.mask_subdir is not None,
    model_params.mask_dim,
    opt_params.feature_init,
    opt_params.static_xyz,
)

optimizer = GaussianOptimizer(
    gaussians,
    opt_params,
    model_params.sh_degree,
    scene.cameras_extent,
    opt_params.static_xyz,
    device,
)

background = torch.tensor([0, 0, 0], dtype=torch.float32, device=device)

## 2. Bucle de Entrenamiento Interactivo

En esta celda puedes ejecutar un número determinado de iteraciones y ver los resultados.

In [ ]:
def get_erank_loss(scaling, eps=1e-8):
    # p_k = s_k^2 / sum(s_j^2)
    s2 = scaling.pow(2)
    p = s2 / (s2.sum(dim=-1, keepdim=True) + eps)
    # Entropía H (Shannon entropy)
    entropy = -(p * torch.log(p + eps)).sum(dim=-1)
    erank = torch.exp(entropy)
    # Barrier loss: penaliza erank cercano a 1 (needle-shaped / 1D)
    # Esto fuerza a que las gaussianas tengan al menos 2 dimensiones útiles
    return -torch.log(erank - 1 + eps).mean()

def get_thinness_loss(scaling, eps=1e-8):
    # Penaliza el inverso de la escala más pequeña (s_min)
    # Evita que se vuelvan láminas 2D sin grosor
    s_min = scaling.min(dim=-1)[0]
    return (1.0 / (s_min + eps)).mean()


In [ ]:
iterations_to_run = 10000
display_interval = 10

losses = []
gaussian_counts = []

start_iter = 1
progress_bar = tqdm(range(opt_params.iterations), desc="Training")
for iteration in progress_bar:
    optimizer.update_learning_rate(iteration)
    
    if iteration % 1000 == 0:
        optimizer.oneup_sh_degree()
        
    viewpoint_cam = next(viewpoint_cams)

    has_depth = hasattr(viewpoint_cam, "original_depth") and viewpoint_cam.original_depth is not None
    render_depth = has_depth and opt_params.lambda_depth > 0
    
    render_pkg = render(
        viewpoint_cam,
        gaussians,
        pipe_params,
        background,
        model_params.sh_degree,
        optimizer.active_sh_degree,
        render_color=True,
        render_depth=render_depth
    )
    
    image = render_pkg.render
    features = render_pkg.features
    variance = render_pkg.variance
    depth = render_pkg.depth

    image, viewspace_point_tensor, visibility_filter, radii = (
        render_pkg.render,
        render_pkg.viewspace_points,
        render_pkg.visibility_filter,
        render_pkg.radii,
    )

    mask_dim = config.model.mask_dim
    gt_image = viewpoint_cam.original_image.to(device)

    loss = torch.tensor(0.0, device=device)

    Ll1 = l1_loss(render_pkg.render, gt_image)
    photometric_loss = (1.0 - config.opt.lambda_dssim) * Ll1 + \
                   config.opt.lambda_dssim * (1.0 - ssim(render_pkg.render, gt_image))
    
    loss += config.opt.photo_lambda * photometric_loss

    depth_loss = None
    if render_depth and depth is not None and has_depth:
        gt_depth = viewpoint_cam.original_depth.to(device)
        depth_loss = l1_loss(depth, gt_depth)

        loss += opt_params.lambda_depth * depth_loss

    if variance is not None and config.opt.var_lambda > 0:
        var_loss = variance[:mask_dim].pow(2).mean()
        loss += config.opt.var_lambda * var_loss

    if viewpoint_cam.masks is not None and features is not None:
        if iteration % config.opt.inst2d_interval == 0:
            gt_masks = viewpoint_cam.masks.to(device)
            inst2d_loss = instance_2d_loss(
                features[:mask_dim],
                gt_masks,
                mask_dim,
                config.opt.inst2d_sample_size,
                config.opt.inst2d_gamma,
                config.opt.inst2d_weights,
                config.opt.inst2d_normalize,
            )
            loss += config.opt.inst2d_lambda * inst2d_loss["total"]

    
    current_scaling = gaussians.get_scaling
    
    warmup_weight = max(0.0, min(1.0, 1 / max(iteration, 1)))

    erank_loss = get_erank_loss(gaussians.get_scaling)
    thin_loss = get_thinness_loss(gaussians.get_scaling)
    
    if warmup_weight > 0:
        loss += warmup_weight * opt_params.lambda_erank * erank_loss
        loss += warmup_weight * opt_params.lambda_thin * thin_loss
    
    optimizer.optimizer.zero_grad(set_to_none=True)
    loss.backward()
    
    # Track statistics for Semantic Pruning
    optimizer.add_importance_stats()
    
    with torch.no_grad():
        losses.append(loss.item())
        optimizer.add_stats(viewspace_point_tensor, visibility_filter, radii)
        
        if iteration > opt_params.densify_from_iter and iteration % opt_params.densification_interval == 0:
            optimizer.densify_and_prune(opt_params.densify_grad_threshold, 0.005, scene.cameras_extent, None)
            
            """
            # Semantic Pruning FeatureSLAM
            if opt_params.semantic_pruning_interval > 0 and iteration % opt_params.semantic_pruning_interval == 0:
                optimizer.semantic_pruning(opt_params.semantic_pruning_percentile)
                # Reset tracking after pruning
                optimizer.importance_accum.fill_(0)
                optimizer.denom.fill_(0)
            """
                
        optimizer.optimizer.step()
        
    progress_bar.set_postfix({
        "L1": f"{Ll1.item():.4f}",
        "Pts": f"{gaussians.get_xyz.shape[0]}",
        "eRank": f"{erank_loss.item():.3f}",
        "depth": f"{depth_loss:.4e}"
    })
            
    losses.append(loss.item())
    gaussian_counts.append(gaussians.num_points)
    
    if iteration % display_interval == 0:
        clear_output(wait=True)
        plt.figure(figsize=(18, 10))
        
        # 1. Output RGB
        plt.subplot(3, 3, 1)
        plt.imshow(image.detach().permute(1, 2, 0).cpu().clamp(0, 1))
        plt.title(f"Render Iter {iteration}")
        plt.axis('off')
        
        # 2. GT RGB
        plt.subplot(3, 3, 2)
        plt.imshow(gt_image.permute(1, 2, 0).cpu().squeeze())
        plt.title("Ground Truth RGB")
        plt.axis('off')


        plt.subplot(3, 3, 3)
        plt.imshow(depth.detach().cpu(), cmap='plasma', vmin=0, vmax=8)
        plt.title(f"Depth Render Iter {iteration}")
        plt.axis('off')


        plt.subplot(3, 3, 4)
        plt.imshow(gt_depth.cpu(), cmap='plasma', vmin=0, vmax=8)
        plt.title("Ground Truth Depth")
        plt.axis('off')
        
        # 3. Ground Truth SAM Mask
        plt.subplot(3, 3, 5)
        # Colorize the integer mask values randomly or with colormap
        mask_np = gt_masks.cpu().numpy()
        plt.imshow(mask_np % 20, cmap='tab20', interpolation='nearest')
        plt.title("GT SAM Mask")
        plt.axis('off')

        # 4. Rendered Features (PCA)
        plt.subplot(3, 3, 6)
        if features is not None:
            # We take only the active mask dimensions (e.g., 8)
            mask_dim = model_params.mask_dim
            feats_to_plot = features[:mask_dim].permute(1, 2, 0).cpu().detach().numpy()
            H, W, C = feats_to_plot.shape
            
            # Compress to RGB using PCA exactly like Demo does
            pca_feats = pca(feats_to_plot.reshape(-1, C), 3, normalize=True).reshape(H, W, 3)
            pca_feats = (pca_feats * 255).astype(np.uint8)
            pca_feats = enhance_image(pca_feats)
            plt.imshow(pca_feats)
        plt.title("Rendered Semantics (PCA)")
        plt.axis('off')
        
        # 5. L1 Loss
        plt.subplot(3, 3, 7)
        plt.plot(losses)
        plt.title("L1 Loss")
        plt.yscale('log')
        
        plt.tight_layout()
        plt.show()
        
        print(f"Iteration {iteration} | Loss: {loss.item():.6f} | Gaussians: {gaussians.num_points} | Depth: {depth_loss:.4e} | w: {warmup_weight:.4e}")

start_iter += iterations_to_run

In [ ]:
from opensplat3d.gaussian_model import save_gaussians

# Guardamos el estado final (usando la iteración actual como nombre)
save_gaussians(gaussians, Path("output/training_study"), iteration)

print(f"Modelo guardado en output/training_study. Listo para ejecutar demo.py")


## 3. Análisis de Puntos

Visualiza cómo ha evolucionado el número de Gaussianas durante el entrenamiento.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(gaussian_counts)
plt.title("Evolución del número de Gaussianas")
plt.xlabel("Iteración")
plt.ylabel("Num Gaussianas")
plt.grid(True)
plt.show()